# Fluid Speech Pro — 25-language transcription on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Fluid-Speech-Pro-multilingual-asr.ipynb)

[`FluidVoice-Labs/fluid-speech-pro`](https://huggingface.co/FluidVoice-Labs/fluid-speech-pro) is FluidVoice Labs' **0.63B** FastConformer-TDT speech recognizer (November 2026). It is built on NVIDIA's [`parakeet-tdt-0.6b-v3`](https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3) (and [`moondream/parakeet-ultra`](https://huggingface.co/moondream/parakeet-ultra)), keeps that model's **25 European languages**, detects the language on its own (**no prompt**), and returns **punctuation and capitalization**.

Why this model: a sub-1B open checkpoint the card ranks as **#2 open** on the [Open ASR Leaderboard](https://huggingface.co/spaces/hf-audio/open_asr_leaderboard) (**6.06** average WER) and the most accurate open model under 1B. The card also reports about **3,700×** real time on one GPU (an hour of audio in about a second). Those are the card's numbers, not a T4 measurement from this notebook.

This notebook uses the **🤗 Transformers** path the card documents:

```python
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="FluidVoice-Labs/fluid-speech-pro")
print(asr("audio.wav")["text"])
```

Needs **Transformers 5.22 or newer**. The card says the Transformers weights leave out the model's one small extra encoder block and decode greedily, and that they give nearly the same results as the NeMo reference in their tests.

**Not in this notebook — timestamps and clean mode.** Word- and segment-level timestamps are documented only for NeMo (`fluid_speech.load` then `transcribe(..., timestamps=True)`). Optional **clean mode** (drop repeated words and fillers such as "um" / "uh") is documented for NeMo, ONNX, and MLX. Installing `nemo_toolkit[asr]` on Colab is a heavy site-packages change — the sibling [Nemotron-Labs Unified ASR notebook](Nemotron-Labs-Unified-ASR-0.6B-multilingual.ipynb) had to pin NeMo from GitHub because PyPI 3.0.0 broke `transcribe`. This notebook stays on the card's Transformers pipeline and skips those NeMo-only calls.

**Licence:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Commercial and non-commercial use are allowed. If you use the model, credit it as **Fluid Speech Pro by FluidVoice**.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM and **no native bfloat16**. `model.safetensors` is about **2.5 GB** of float32 weights. This notebook loads **float16** on T4. Set `DTYPE = torch.float32` in the GPU cell and re-run load if a transcript looks degraded.

**Use cases:** [when to pick this model](https://github.com/donvito/notebooks/blob/main/docs/Fluid-Speech-Pro-multilingual-asr.md)

**Links:** [model card](https://huggingface.co/FluidVoice-Labs/fluid-speech-pro) · [announcement](https://x.com/FluidVoiceApp/status/2121308183314796772) · [Open ASR Leaderboard](https://huggingface.co/spaces/hf-audio/open_asr_leaderboard) · [FluidVoice blog](https://fluidvoice.com/fluid/blog/fluidspeech-pro)

In [ ]:
%pip install -q -U "transformers>=5.22.0" datasets soundfile

# Card install is: pip install -U transformers  (needs 5.22+ for ParakeetForTDT).
# Do NOT pip install torch here. Colab already ships a CUDA wheel; forcing a
# reinstall often drops GPU support or pulls a CPU-only build.
# datasets + soundfile load the LibriSpeech dummy clip and one FLEURS example.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.

## 1. Check the GPU

Confirm CUDA is available, print the device and free VRAM, and pick a dtype. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses. Fall back to float32 if outputs degrade.

In [ ]:
import os
import time

import torch
import transformers
from packaging.version import Version

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

assert Version(transformers.__version__) >= Version("5.22.0"), (
    f"transformers {transformers.__version__} is loaded; Fluid Speech Pro needs 5.22+ "
    "(card: ParakeetForTDT pipeline). Run the install cell, then Runtime > Restart session."
)

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

major, minor = torch.cuda.get_device_capability()
DEVICE = 0
# T4 (sm_75) has no native bfloat16. Checkpoint dtype is float32.
# float16 is the T4 path. Set DTYPE = torch.float32 if transcripts look degraded.
DTYPE = torch.float16
MODEL_ID = "FluidVoice-Labs/fluid-speech-pro"
TARGET_SR = 16_000

free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"dtype: {DTYPE}  (saved weights are float32; T4 has no native bf16)")
print(f"VRAM free: {free_b / 1024**3:.1f} GB of {total_b / 1024**3:.1f} GB")
if major < 8:
    print("No native bfloat16 on this GPU — staying on float16.")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB | "
        f"free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 2. Load the model

The card's call is `pipeline("automatic-speech-recognition", model="FluidVoice-Labs/fluid-speech-pro")`. We only add `device=0` and `dtype=...` so the T4 runs float16 instead of defaulting to CPU / float32. First run downloads about **2.5 GB** (`model.safetensors`). No `trust_remote_code` — `config.json` names native `ParakeetForTDT`.

In [ ]:
from transformers import pipeline

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
asr = pipeline(
    "automatic-speech-recognition",
    model=MODEL_ID,
    device=DEVICE,
    dtype=DTYPE,
)
torch.cuda.synchronize()
print(f"loaded {MODEL_ID} in {time.perf_counter() - t0:.1f}s")
print("class:", type(asr.model).__name__)
print("dtype:", next(asr.model.parameters()).dtype, "| device:", next(asr.model.parameters()).device)
n_params = sum(p.numel() for p in asr.model.parameters())
print(f"parameters: {n_params / 1e6:.0f}M  (card: 0.63B)")
print_vram("after load")
print()
print("If greedy transcripts later look empty or garbled, reload with DTYPE = torch.float32.")


## 3. Demo 1 — English clip with punctuation

A short LibriSpeech *clean* excerpt from [`hf-internal-testing/librispeech_asr_dummy`](https://huggingface.co/datasets/hf-internal-testing/librispeech_asr_dummy) (73 clips, ~9 MB). These are public-domain LibriVox audiobook readings, the same family as OpenSLR LibriSpeech (CC BY 4.0). This is **not** the official `test-clean` leaderboard split.

The dummy reference is uppercase and has no punctuation. Fluid Speech Pro's card says output includes punctuation and capitalization. The call is exactly the card's `asr("audio.wav")["text"]`.

In [ ]:
from pathlib import Path

import numpy as np
import soundfile as sf
from datasets import Audio, load_dataset
from IPython.display import Audio as play, display

SAMPLE_DIR = Path("fluid_speech_samples")
SAMPLE_DIR.mkdir(exist_ok=True)


def write_wav(path, array, sampling_rate):
    path = Path(path)
    audio = np.asarray(array, dtype=np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=-1)
    sf.write(path, audio, int(sampling_rate))
    return path, len(audio) / float(sampling_rate)


ds = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean",
    split="validation",
)
ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))

clip = ds[0]
en_path, en_seconds = write_wav(
    SAMPLE_DIR / "en-librispeech.wav",
    clip["audio"]["array"],
    clip["audio"]["sampling_rate"],
)

print(f"id: {clip['id']}  |  {en_seconds:.2f}s  |  sr={clip['audio']['sampling_rate']}")
print("reference (no punctuation):", clip["text"])
display(play(str(en_path)))

torch.cuda.synchronize()
t0 = time.perf_counter()
en_out = asr(str(en_path))
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
print()
print("transcript:", en_out["text"])
print(f"wall time: {elapsed:.2f}s  |  audio: {en_seconds:.2f}s")
print_vram("after English clip")


## 4. Demo 2 — German FLEURS clip (automatic language detection)

One test example from [`google/fleurs`](https://huggingface.co/datasets/google/fleurs) `de_de` (CC BY 4.0), streamed so Colab does not download the full German split. German is one of the 25 European languages on the card.

The Transformers call takes **no language argument**. The card: the model "detects the language on its own, with no prompt needed." This cell does not print a language-ID code — the pipeline returns text only — it just transcribes German audio with the same `asr("audio.wav")` call as English.

In [ ]:
fleurs = load_dataset("google/fleurs", "de_de", split="test", streaming=True)
example = next(iter(fleurs))
de_audio = example["audio"]
de_path, de_seconds = write_wav(
    SAMPLE_DIR / "de-fleurs.wav",
    de_audio["array"],
    de_audio["sampling_rate"],
)

print("FLEURS language:", example.get("language"), "| lang_id:", example.get("lang_id"))
print(f"audio: {de_seconds:.2f}s  |  sr={de_audio['sampling_rate']}")
print("reference:", example.get("raw_transcription") or example.get("transcription"))
display(play(str(de_path)))

torch.cuda.synchronize()
t0 = time.perf_counter()
de_out = asr(str(de_path))
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
print()
print("transcript:", de_out["text"])
print(f"wall time: {elapsed:.2f}s  |  audio: {de_seconds:.2f}s")
print("(No language prompt was passed — same pipeline call as Demo 1.)")
print_vram("after German clip")


## 5. Demo 3 — longer clip: audio seconds vs wall time

Concatenate several dummy LibriSpeech clips into one file and transcribe it with the same `asr("audio.wav")` call. This is a **T4 wall-clock** check on a short teaching file, not the card's ~3,700× figure (that number is from the card, on "one GPU", with their batching).

**Timestamps:** skipped. The card documents word/segment timestamps on NeMo (`out.timestamp["word"]`), not on this Transformers pipeline.

In [ ]:
n_clips = 8
pieces = []
for row in ds.select(range(n_clips)):
    arr = np.asarray(row["audio"]["array"], dtype=np.float32)
    if arr.ndim > 1:
        arr = arr.mean(axis=-1)
    pieces.append(arr)

# 0.25 s of silence between clips so the join is audible, not a hard cut.
gap = np.zeros(int(0.25 * TARGET_SR), dtype=np.float32)
long_audio = pieces[0]
for piece in pieces[1:]:
    long_audio = np.concatenate([long_audio, gap, piece])

long_path, long_seconds = write_wav(SAMPLE_DIR / "en-longer.wav", long_audio, TARGET_SR)
print(f"concatenated {n_clips} dummy clips → {long_path.name} ({long_seconds:.2f}s)")
display(play(str(long_path)))

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
long_out = asr(str(long_path))
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
rtf = elapsed / long_seconds if long_seconds else float("nan")

print()
print("transcript:", long_out["text"])
print(f"audio: {long_seconds:.2f}s  |  wall time: {elapsed:.2f}s  |  RTF {rtf:.3f}x")
print("(RTF < 1 means faster than real time. This is T4 + short-file overhead, not the card's 3,700×.)")
print_vram("after longer clip")


## 6. Caveats and cleanup

**What this notebook is for.** Offline multilingual transcription on a free T4 with the card's 🤗 Transformers pipeline: English punctuation, a non-English European clip with no language prompt, and a longer-file speed check.

**What it is not.** NeMo timestamps, NeMo/ONNX/MLX clean mode, streaming, speech translation, or speaker labels. Pair with [Nemotron-3 Diarization](Nemotron-3-Diarization.ipynb) if you need who-spoke-when.

**Languages.** bg, hr, cs, da, nl, en, et, fi, fr, de, el, hu, it, lv, lt, mt, pl, pt, ro, sk, sl, es, sv, ru, uk. The card does not list non-European languages.

**T4 notes.** float16 weights on T4; no bfloat16. Keep single clips to a few minutes; for long recordings, split first. The card asks you to check the model on your own audio, accents, and languages before you rely on it, and notes more mistakes on heavy noise, overlapping speakers, strong accents, and rare names.

**Related notebooks:** [Nemotron-Labs Unified ASR 0.6B](Nemotron-Labs-Unified-ASR-0.6B-multilingual.ipynb) (NeMo, 25 European languages, timestamps), [Granite Speech 5.0 2B](granite-speech-5.0-2b-asr-translation.ipynb) (ASR + translation), [oruk-loop-153M](oruk-loop-153M-looped-asr.ipynb) (English ASR), [Whisper](AIBackends-Whisper-audio-transcription.ipynb).

Free the GPU before you open another notebook:

In [ ]:
import gc

print_vram("before cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
del asr
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
